In [3]:
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (AutoTokenizer, AutoConfig,
    AutoModelForSequenceClassification, Trainer, TrainingArguments,
    DataCollatorWithPadding)
from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import torch
import numpy as np
import evaluate
import mlflow
import mlflow.transformers
import os

In [4]:
# MLflow Setup
# Configure MLflow tracking
mlflow.set_tracking_uri("file:./mlruns")  # Local tracking in ./mlruns directory

# Create or get experiment
experiment_name = "DistilBERT-LoRA-Sentiment-Analysis"
try:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=None  # Local filesystem
    )
    print(f"Created new experiment: {experiment_name} with ID: {experiment_id}")
except:
    # Experiment already exists, get its ID
    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {experiment_name} with ID: {experiment_id}")

mlflow.set_experiment(experiment_name)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Access MLflow UI: mlflow ui --backend-store-uri {mlflow.get_tracking_uri()} --port 8000")

Created new experiment: DistilBERT-LoRA-Sentiment-Analysis with ID: 622340514322182606
MLflow tracking URI: file:./mlruns
Access MLflow UI: mlflow ui --backend-store-uri file:./mlruns --port 8000


In [5]:
model_checkpoint = "distilbert-base-uncased"

id2label = {0: 'Negative', 1: 'Positive'}
label2id = {'Negative': 0, 'Positive': 1}

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels = 2, id2label=id2label, label2id=label2id)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
dataset = load_dataset("shawhin/imdb-truncated")
print(dataset)

README.md:   0%|          | 0.00/592 [00:00<?, ?B/s]

data/train-00000-of-00001-5a744bf76a1d84(…):   0%|          | 0.00/836k [00:00<?, ?B/s]

data/validation-00000-of-00001-a3a52fabb(…):   0%|          | 0.00/853k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 1000
    })
})


In [7]:
# create tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# create tokenize function
def tokenize_function(examples):
    text = examples["text"]

    # tokenize and truncate the text if it is too long
    tokenizer.truncation_side = 'left'
    tokenized_inputs = tokenizer(text, return_tensors='np', truncation=True, max_length=512)

    return tokenized_inputs

# add pad token if None exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

# tokenize the training and validation sets
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})


In [8]:
# create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

# define evaluation function to pass into trainer later
def compute_metrics(p):
    predictions, labels = p
    # Use argmax for other metrics that require class predictions
    predictions = np.argmax(predictions, axis=1)

    # Calculate metrics
    accuracy_score = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    precision_score = precision.compute(predictions=predictions, references=labels)["precision"]
    recall_score = recall.compute(predictions=predictions, references=labels)["recall"]
    f1_score = f1.compute(predictions=predictions, references=labels)["f1"]

    metrics_dict = {
        "accuracy": accuracy_score,
        "precision": precision_score,
        "recall": recall_score,
        "f1": f1_score
    }

    # Log metrics to MLflow if there's an active run
    if mlflow.active_run():
        # Log metrics with step to track progress over time
        for metric_name, metric_value in metrics_dict.items():
            mlflow.log_metric(f"eval_{metric_name}", metric_value)

    return metrics_dict

In [10]:
# Untrained model predictions/performance

text_list = ["It was good.", "Not a fan, don't recommend.",
             "Better than the first one.", "This is not worth watching even once.",
             "This one is a pass."]

print("Untrained model predictions:")
print("---------------------------")

# Get the device of the model
device = next(model.parameters()).device

for text in text_list:
    # tokenize the text and move to the same device as the model
    inputs = tokenizer.encode(text, return_tensors='pt').to(device)
    # compute logits
    logits =  model(inputs).logits
    # convert logits to labels
    predictions = torch.argmax(logits)

    print(text + " - " + id2label[predictions.tolist()])

Untrained model predictions:
---------------------------
It was good. - Negative
Not a fan, don't recommend. - Negative
Better than the first one. - Negative
This is not worth watching even once. - Negative
This one is a pass. - Negative


In [11]:
peft_config = LoraConfig(task_type='SEQ_CLS', # sequence classification
                         r = 4, # intrinsic rank of trainable weight matrix
                         lora_alpha=32,  # this is like learning rate
                         lora_dropout=0.01, # probability of dropout
                         target_modules=['q_lin'])  # we apply lora to query layer

In [12]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 628,994 || all params: 67,584,004 || trainable%: 0.9307


In [13]:
# hyperparameters
lr = 1e-4
batch_size = 8
num_epochs = 10

# define training arguments
training_args = TrainingArguments(output_dir=model_checkpoint + '-lora-text-classification',
                learning_rate=lr,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                num_train_epochs=num_epochs,
                weight_decay=0.01,
                eval_strategy='epoch',
                save_strategy='epoch',
                load_best_model_at_end=True,
                report_to="none")

In [14]:
# Start MLflow run and log parameters
run_name = f"LoRA_r{peft_config.r}_alpha{peft_config.lora_alpha}_lr{lr}"
mlflow.start_run(run_name=run_name)

# Log model info
mlflow.log_param("model_checkpoint", model_checkpoint)
mlflow.log_param("task", "sentiment_classification")

# Log dataset info
mlflow.log_params({
    "dataset": "shawhin/imdb-truncated",
    "train_size": len(tokenized_dataset["train"]),
    "val_size": len(tokenized_dataset["validation"])
})

# Log LoRA configuration
mlflow.log_params({
    "lora_r": peft_config.r,
    "lora_alpha": peft_config.lora_alpha,
    "lora_dropout": peft_config.lora_dropout,
    "lora_target_modules": str(peft_config.target_modules)
})

# Log training hyperparameters
mlflow.log_params({
    "learning_rate": lr,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "weight_decay": training_args.weight_decay,
    "warmup_ratio": training_args.warmup_ratio if hasattr(training_args, "warmup_ratio") else 0,
    "max_length": 512
})

print(f"Started MLflow run: {run_name}")
print(f"Run ID: {mlflow.active_run().info.run_id}")

Started MLflow run: LoRA_r4_alpha32_lr0.0001
Run ID: 5a9eb7daab564bda820c01b1d8d838f3


In [15]:
# create trainer object
trainer = Trainer(model=model,
                  args=training_args,
                  data_collator=data_collator,
                  train_dataset=tokenized_dataset['train'],
                  eval_dataset=tokenized_dataset['validation'],
                  tokenizer=tokenizer,
                  compute_metrics=compute_metrics)

# Train the model and capture training results
print("Starting training...")
train_result = trainer.train()

# Log training metrics
training_metrics = {
    "train_loss": train_result.training_loss,
    "train_runtime_seconds": train_result.metrics["train_runtime"],
    "train_samples_per_second": train_result.metrics["train_samples_per_second"],
    "train_steps_per_second": train_result.metrics["train_steps_per_second"]
}

# Log to MLflow
for metric_name, metric_value in training_metrics.items():
    mlflow.log_metric(metric_name, metric_value)

print(f"Training completed in {training_metrics['train_runtime_seconds']:.2f} seconds")
print(f"Final training loss: {training_metrics['train_loss']:.4f}")

/tmp/ipython-input-2778012548.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.392215,0.872000,0.850943,0.902000,0.875728
2,No log,0.268341,0.885000,0.924945,0.838000,0.879328
3,No log,0.259322,0.895000,0.922912,0.862000,0.891417
4,0.377900,0.258828,0.903000,0.917184,0.886000,0.901322
5,0.377900,0.289439,0.895000,0.934066,0.850000,0.890052
6,0.377900,0.274138,0.908000,0.921488,0.892000,0.906504
7,0.377900,0.279508,0.906000,0.921162,0.888000,0.904277
8,0.228000,0.281776,0.907000,0.907816,0.906000,0.906907
9,0.228000,0.287686,0.907000,0.914460,0.898000,0.906155
10,0.228000,0.289160,0.906000,0.914286,0.896000,0.905051


Training completed in 472.83 seconds
Final training loss: 0.2820


In [16]:
# Final evaluation and model logging
print("Running final evaluation...")
eval_results = trainer.evaluate()

# Log final evaluation metrics
print("\nFinal Evaluation Results:")
for key, value in eval_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
        mlflow.log_metric(f"final_{key}", value)

# Save and log the model
model_info = {
    "model_type": "LoRA fine-tuned DistilBERT",
    "base_model": model_checkpoint,
    "task": "sentiment_classification",
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in model.parameters()),
}

# Log model metadata
mlflow.log_params(model_info)

# Log the model to MLflow
model_path = "model"
mlflow.transformers.log_model(
    transformers_model={
        "model": model,
        "tokenizer": tokenizer
    },
    artifact_path=model_path,
    task="text-classification",
    registered_model_name="distilbert-lora-sentiment"
)

print(f"\nModel logged to MLflow at path: {model_path}")
print(f"Model can be loaded with: mlflow.transformers.load_model('runs:/<run_id>/{model_path}')")

Running final evaluation...


2025/08/17 05:11:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Final Evaluation Results:
eval_loss: 0.2588
eval_accuracy: 0.9030
eval_precision: 0.9172
eval_recall: 0.8860
eval_f1: 0.9013
eval_runtime: 14.5029
eval_samples_per_second: 68.9520
eval_steps_per_second: 8.6190
epoch: 10.0000


Device set to use cuda:0
2025/08/17 05:11:39 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2025/08/17 05:11:39 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository distilbert-base-uncased will be logged instead.


README.md:   0%|          | 0.00/8.58k [00:00<?, ?B/s]

LICENSE:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

2025/08/17 05:11:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/08/17 05:11:40 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/08/17 05:11:40 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into memory, we don't infer the pip requirement for the model. Instead, we will 


Model logged to MLflow at path: model
Model can be loaded with: mlflow.transformers.load_model('runs:/<run_id>/model')


Successfully registered model 'distilbert-lora-sentiment'.
Created version '1' of model 'distilbert-lora-sentiment'.


In [17]:
print("Trained model predictions:")
print("-------------------------")

device = next(model.parameters()).device

# Store predictions for logging
sample_predictions = []

for text in text_list:
    # tokenize the text
    inputs = tokenizer.encode(text, return_tensors='pt').to(device)
    # compute logits
    logits = model(inputs).logits
    # convert logits to labels
    predictions = torch.argmax(logits, 1)

    predicted_label = id2label[predictions.tolist()[0]]
    confidence = torch.softmax(logits, dim=1).max().item()

    print(f"{text} - {predicted_label} (confidence: {confidence:.4f})")

    sample_predictions.append({
        "text": text,
        "predicted_label": predicted_label,
        "confidence": confidence
    })

# Log sample predictions to MLflow
import json
import tempfile

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(sample_predictions, f, indent=2)
    predictions_path = f.name

mlflow.log_artifact(predictions_path, "sample_predictions")
os.unlink(predictions_path)  # Clean up temp file

# End the MLflow run
mlflow.end_run()

print("\nMLflow tracking completed!")
print(f"View the results by running: mlflow ui --backend-store-uri {mlflow.get_tracking_uri()} --port 8000")
print("Then open http://localhost:8000 in your web browser")

Trained model predictions:
-------------------------
It was good. - Positive (confidence: 0.6684)
Not a fan, don't recommend. - Negative (confidence: 0.8033)
Better than the first one. - Positive (confidence: 0.6932)
This is not worth watching even once. - Negative (confidence: 0.6525)
This one is a pass. - Positive (confidence: 0.5209)

MLflow tracking completed!
View the results by running: mlflow ui --backend-store-uri file:./mlruns --port 8000
Then open http://localhost:8000 in your web browser


In [18]:
# How to load and use the model from MLflow (optional)

# Get the latest run ID from the experiment
def get_latest_model_uri():
    experiment = mlflow.get_experiment_by_name("DistilBERT-LoRA-Sentiment-Analysis")
    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["start_time DESC"], max_results=1)
        if len(runs) > 0:
            run_id = runs.iloc[0].run_id
            return f"runs:/{run_id}/model"
    return None

# Example code to load the model (uncomment to use)
"""
model_uri = get_latest_model_uri()
if model_uri:
    print(f"Loading model from: {model_uri}")
    # Load the model
    loaded_model = mlflow.transformers.load_model(model_uri)

    # Use the loaded model for inference
    test_text = "This movie was absolutely fantastic!"
    result = loaded_model(test_text)[0]
    predicted_label = result['label']
    score = result['score']
    print(f"Prediction: {predicted_label}, Score: {score:.4f}")
else:
    print("No model found. Run the training first.")
"""

print("MLflow integration completed successfully!")
print("\nYou can now:")
print("1. View experiments by running: mlflow ui --backend-store-uri file:./mlruns --port 8000")
print("2. Compare different runs and hyperparameters")
print("3. Download or load your trained models")
print("4. Export models for deployment")

MLflow integration completed successfully!

You can now:
1. View experiments by running: mlflow ui --backend-store-uri file:./mlruns --port 8000
2. Compare different runs and hyperparameters
3. Download or load your trained models
4. Export models for deployment
